# Homework 6: Decision Trees and Ensemble Learning

## Setup

In [4]:
import hashlib

import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor

DATA_PATH = 'data/car_fuel_efficiency_2026.csv'

with open(DATA_PATH, 'rb') as f_in:
    print('sha256:', hashlib.sha256(f_in.read()).hexdigest())

print('sklearn:', sklearn.__version__)
print('xgboost:', xgb.__version__)

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

sha256: 00a5cab178a8b7cd6e9157ee71dabc236ba7f20b1832336d358b07af14ba431f
sklearn: 1.9.1
xgboost: 3.2.0
(10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Prepare and split the dataset

In [5]:
TARGET = 'fuel_efficiency_mpg'

print('missing values before:\n', df.isnull().sum()[df.isnull().sum() > 0])
df = df.fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

y_train = df_train[TARGET].values
y_val = df_val[TARGET].values

df_train = df_train.drop(columns=[TARGET]).reset_index(drop=True)
df_val = df_val.drop(columns=[TARGET]).reset_index(drop=True)

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

features = list(dv.get_feature_names_out())
print(len(df_train), len(df_val))
print(features)


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def original_name(feature):
    # 'origin=USA' -> 'origin'
    return feature.split('=')[0]

missing values before:
 horsepower      877
acceleration    264
dtype: int64
6000 2000
['acceleration', 'drivetrain=All-wheel drive', 'drivetrain=Front-wheel drive', 'drivetrain=Rear-wheel drive', 'engine_displacement', 'fuel_type=Diesel', 'fuel_type=Gasoline', 'fuel_type=Hybrid', 'horsepower', 'model_year', 'num_cylinders', 'num_doors', 'origin=Asia', 'origin=Europe', 'origin=USA', 'vehicle_weight']


## Q1. Splitting feature

In [6]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=features))

root_feature = features[dt.tree_.feature[0]]
print('one-hot feature:', root_feature)
print('original column:', original_name(root_feature))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]

one-hot feature: model_year
original column: model_year


## Q2. Random forest RMSE

In [7]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

rmse_q2 = rmse(y_val, rf.predict(X_val))
print(round(rmse_q2, 4))

options = [0.1837, 1.837, 18.37, 183.7]
print('closest option:', min(options, key=lambda o: abs(o - rmse_q2)))

1.8394
closest option: 1.837


## Q3. Number of estimators

In [8]:
scores_q3 = {}
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    scores_q3[n] = round(rmse(y_val, rf.predict(X_val)), 3)
    print(n, scores_q3[n])

# on a tie at 3 decimals, take the smaller n
best_rmse = min(scores_q3.values())
print('best n_estimators:', min(n for n, s in scores_q3.items() if s == best_rmse))

10 1.839
50 1.773
100 1.768
150 1.768
best n_estimators: 100


## Q4. Best max_depth

In [9]:
mean_scores = {}
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
            n_estimators=n, max_depth=d, random_state=1, n_jobs=-1
        )
        rf.fit(X_train, y_train)
        scores.append(rmse(y_val, rf.predict(X_val)))
    mean_scores[d] = np.mean(scores)
    print(d, round(mean_scores[d], 4), [round(s, 3) for s in scores])

print('best max_depth:', min(mean_scores, key=mean_scores.get))

10 1.7541 [np.float64(1.782), np.float64(1.747), np.float64(1.744), np.float64(1.744)]
15 1.7837 [np.float64(1.83), np.float64(1.771), np.float64(1.767), np.float64(1.767)]
20 1.7878 [np.float64(1.84), np.float64(1.774), np.float64(1.769), np.float64(1.768)]
25 1.7867 [np.float64(1.84), np.float64(1.773), np.float64(1.767), np.float64(1.767)]
best max_depth: 10


## Q5. Feature importance

In [10]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=features)
print(importances.sort_values(ascending=False).head(10))

# sum one-hot features back into their original columns
imp_orig = importances.groupby(original_name).sum()
candidates = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
print(imp_orig[candidates].sort_values(ascending=False))
print('most important:', imp_orig[candidates].idxmax())

model_year                      0.295915
vehicle_weight                  0.203067
fuel_type=Gasoline              0.160712
horsepower                      0.078468
engine_displacement             0.061152
acceleration                    0.055342
origin=USA                      0.042268
drivetrain=Front-wheel drive    0.020108
num_doors                       0.019364
fuel_type=Diesel                0.018045
dtype: float64
vehicle_weight         0.203067
horsepower             0.078468
engine_displacement    0.061152
acceleration           0.055342
dtype: float64
most important: vehicle_weight


## Q6. XGBoost eta

In [11]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

results_q6 = {}
for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }
    evals_result = {}
    model = xgb.train(
        xgb_params, dtrain, num_boost_round=100,
        evals=watchlist, evals_result=evals_result, verbose_eval=False
    )
    val_curve = evals_result['val']['rmse']
    results_q6[eta] = round(rmse(y_val, model.predict(dval)), 3)
    print(f'eta={eta}: RMSE after 100 rounds = {results_q6[eta]}, '
          f'curve minimum = {min(val_curve):.3f} (round {int(np.argmin(val_curve)) + 1})')

if results_q6[0.3] == results_q6[0.1]:
    print('answer: Both give equal value')
else:
    print('answer:', min(results_q6, key=results_q6.get))

eta=0.3: RMSE after 100 rounds = 1.826, curve minimum = 1.732 (round 11)
eta=0.1: RMSE after 100 rounds = 1.725, curve minimum = 1.712 (round 40)
answer: 0.1
